# Lesson 9 — Neural Networks

Self-assessment. No code: every answer is a sentence, a short derivation, or a
diagnosis.

Numbers quoted throughout come from the lesson's handout and notebooks:
3,000 Meridian Instruments sensors judged on two calibration axes, 2,250 of
them for training, with a test rig that records the wrong verdict for 3% of
units and so caps every accuracy in this lesson at **0.97**; 800 two-channel
drift units in four clouds of 200, the exclusive-or (XOR) problem in
disguise; and the 1,797 handwritten 8&times;8 digit images bundled with
scikit-learn, in ten classes.

As in earlier lessons, several questions ask you to *derive* or *criticise* a
result rather than recall it &mdash; those are the ones worth your time. Three
of them are about how a number was measured rather than what it was, which is
half of what this lesson teaches.


## Part 1 — One neuron is one line


**1. Define the perceptron, state how it relates to the logistic-regression
unit of lesson 4, and say precisely which set of points a single unit is
undecided about.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The <b>perceptron</b> (Rosenblatt, 1958) has a weight vector w and a bias b, and outputs 1 when <code>w<sup>T</sup>x + b &gt; 0</code> and 0 otherwise &mdash; a hard threshold on a weighted sum of the inputs.</li>
        <li>Replace that hard threshold with the sigmoid <code>&sigma;(z) = 1/(1 + e<sup>&minus;z</sup>)</code> and you have exactly lesson 4's logistic regression unit, <code>&#375; = &sigma;(w<sup>T</sup>x + b)</code>. Nothing about the unit is new; what is new is treating it as a <b>component</b> that a network stacks rather than as a whole model.</li>
        <li>The undecided set is <code>&#375; = 0.5</code>, which is <code>w<sup>T</sup>x + b = 0</code>: a straight line in two dimensions, a plane in three, a hyperplane in general. So <b>a neuron is a line</b>, and everything one neuron can express is &ldquo;which side of this line are you on, and how far&rdquo;.</li>
    </ul>
    </p>
</details>


**2. Explain why logistic regression fitted to the acceptance data scores
0.5507 on the test set when always predicting &ldquo;accepted&rdquo; scores
0.5480 &mdash; and why this is not an optimiser that failed.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The fitted coefficients are (0.0096, 0.0701) with intercept 0.1892, which is almost exactly the constant model: the fit barely tilts the boundary at all.</li>
        <li>The reason is <b>symmetry</b>. The accept region is a disc centred on the origin once each axis is standardised, so for every accepted unit at (z&#8321;, z&#8322;) there is on average a matching one at (&minus;z&#8321;, &minus;z&#8322;). Any line that gains accuracy on one side gives back the same amount on the other, so no tilt pays.</li>
        <li>0.5507 is therefore the <b>correct answer to the question a line is able to ask</b>, not a convergence failure. A different optimiser, more iterations or a better learning rate would change nothing; the shape of the boundary is the constraint.</li>
    </ul>
    </p>
</details>


**3. Both of Meridian's tolerances are 1.25 production spreads wide. Derive
the fraction of units inside the accept region for a standard two-dimensional
normal, and say why standardising each axis is the step that makes the shape
visible.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The gain tolerance is 0.50 dB against a production spread of 0.40, and the phase tolerance 3.75&deg; against a spread of 3.00 &mdash; both exactly 1.25 spreads. Dividing each axis by its own spread turns an ellipse in mixed units (decibels against degrees) into a <b>circle of radius 1.25</b> in dimensionless units.</li>
        <li>For a standard two-dimensional normal the squared distance from the origin is exponentially distributed, giving <code>P(&#8214;z&#8214; &lt; s) = 1 &minus; e<sup>&minus;s&#178;/2</sup></code>. At s = 1.25: <code>1 &minus; e<sup>&minus;0.78125</sup> = 1 &minus; 0.4578 = 0.5422</code>, against 0.5497 measured on the 3,000 generated units.</li>
        <li>Without standardisation the two axes are incomparable numbers and the region looks like an arbitrary ellipse; after it, the rule is one number &mdash; a radius &mdash; and the reason no line can enclose it is immediate.</li>
    </ul>
    </p>
</details>


**4. A brute-force search over 72,762 candidate straight lines found one
scoring 0.6880, while the honest figure for the population is 0.6491. Explain
why the first number is optimistic and name the mechanism.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>0.6880 is the winner's score <b>on the same data that was used to choose it</b>. Each candidate's measured score is its true accuracy plus sampling noise; taking the maximum over 72,762 candidates selects for large positive noise as much as for genuine accuracy, so the winner's measured score is biased upward.</li>
        <li>This is <b>selection bias</b> &mdash; lesson 5's central point, arriving in a form that looks entirely innocent. Nothing here resembles the obvious sin of training on the test set; the test set was merely used to pick one model out of many, which is the same sin.</li>
        <li>The gap is <b>4 percentage points</b>: 0.6880 against the 0.6491 that section 11 computes by integrating over the population, with no test set involved at all. The honest protocol is to select on one split and report on another that nothing was chosen against.</li>
    </ul>
    </p>
</details>


## Part 2 — A hidden layer: fences made of lines


**5. Meridian's two-channel drift problem is the exclusive-or (XOR) function
&mdash; a unit is correctable exactly when its two drifts share a sign. Show
that the rule depends only on |a + b|, and explain why that puts it beyond any
single line.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Coding the two channels as &plusmn;1, &ldquo;the signs agree&rdquo; means <code>ab = 1</code>, which means <code>a + b = &plusmn;2</code>; &ldquo;the signs differ&rdquo; means <code>a + b = 0</code>. So the verdict is entirely a question about <code>|a + b|</code> being large or small.</li>
        <li>&ldquo;Large in absolute value&rdquo; is the <b>outside of a strip</b>, and a strip has two boundaries. One line gives you one boundary, so a single unit cannot express it no matter how its weights are set.</li>
        <li>Measured, logistic regression on the 800 drift units scores exactly <b>0.5000</b> &mdash; a line is worth literally nothing here, not merely a little less than a network.</li>
    </ul>
    </p>
</details>


**6. State the hand-built two-unit network of handout section 3.2, work the
four cloud centres through it, and say what it scores untrained.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With <code>W<sup>[1]</sup> = [[1, &minus;1], [1, &minus;1]]</code>, <code>b<sup>[1]</sup> = (&minus;1, &minus;1)</code>, <code>W<sup>[2]</sup> = (10, 10)</code> and <code>b<sup>[2]</sup> = &minus;1</code>, the two hidden units apply the <b>rectified linear unit (ReLU)</b>, <code>ReLU(z) = max(0, z)</code>, to a shifted sum: <code>h&#8321; = ReLU(a + b &minus; 1)</code> and <code>h&#8322; = ReLU(&minus;a &minus; b &minus; 1)</code>.</li>
        <li>Between them the pair computes a clipped <code>|a + b|</code>: h&#8321; fires only when a + b &gt; 1 and h&#8322; only when a + b &lt; &minus;1. At the centres (+1,+1) and (&minus;1,&minus;1) the output pre-activation is 10&times;1 &minus; 1 = 9, so <code>&#375; = 0.9999</code>; at (+1,&minus;1) and (&minus;1,+1) both hidden units are off, the pre-activation is &minus;1 and <code>&#375; = 0.2689</code>. All four rows are right, and the network's decision rule is exactly <code>|a + b| &gt; 1.1</code>.</li>
        <li>Run over all 800 units with their noise, and <b>nothing trained</b>, it scores <b>0.9938</b> &mdash; five units wrong out of eight hundred. The weights were reasoned out, not fitted, which tells you what training is <i>searching for</i>.</li>
    </ul>
    </p>
</details>


**7. What did the hidden layer actually do to the drift data? Answer in
terms of the hidden-space figure, and name the idea.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It <b>classified nothing</b>. In the hidden units' coordinates the two correctable clouds sit at (1, 0) and (0, 1) while both uncorrectable clouds are stacked at the origin &mdash; and in those coordinates a single line separates them.</li>
        <li>So the hidden layer <b>moved the data until the last layer's line was enough</b>. The useful thing a network learns is not the final boundary but the coordinates in which the final boundary is simple. The name for this is <b>representation learning</b>.</li>
        <li>Every deep architecture is this observation applied repeatedly; lesson 10's convolutional network is the same trick with a restriction on which weights are allowed to be non-zero.</li>
    </ul>
    </p>
</details>


**8. Two hidden units suffice for the drift problem, yet across 20 restarts
at width 2 the median accuracy is 0.7500 and only 4 of 20 runs beat 0.95.
Explain what 0.7500 corresponds to and what the sweep shows about width.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>0.7500 is exactly three of the four clouds: the network has used both of its lines to carve off a single quadrant instead of building the strip. It is a <b>local minimum</b>, and from most starting points it is downhill &mdash; which is why it is the median outcome rather than a rarity.</li>
        <li>Widening fixes it without adding any capacity the problem needs: 3 units give 15/20 runs above 0.95, 4 units 19/20, and 8 units 20/20, all with median accuracy 1.0000.</li>
        <li>The moral is that <b>representable and findable are different properties</b>. The extra units in a production network are usually not extra capacity but extra starting points, so that some unit begins near a useful line.</li>
    </ul>
    </p>
</details>


**9. State the universal approximation theorem in the form due to Cybenko
and Hornik, and list what it does not promise.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A network with <b>one</b> hidden layer, a non-polynomial activation and enough units can approximate any continuous function on a bounded region to any accuracy you like.</li>
        <li>It says only that such a network <b>exists</b>. It says nothing about how many units &ldquo;enough&rdquo; is, nothing about whether any training procedure will find it, and nothing about how the network behaves outside the bounded region.</li>
        <li>Question 8 is a two-unit counterexample to the usual misreading: the approximation existed and plain gradient descent missed it <b>16 times in 20</b>. The theorem's real content is a licence to stop worrying about expressiveness and start worrying about optimisation and data.</li>
    </ul>
    </p>
</details>


## Part 3 — Forward propagation, and what the shapes tell you


**10. Give the shape of every array in a one-hidden-layer network of H units
on m examples with n inputs, and state the two structural facts that follow
from those shapes.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>X</code> is m&times;n (one example per <b>row</b>, as scikit-learn and Keras both do); <code>W<sup>[1]</sup></code> is n&times;H; <code>b<sup>[1]</sup></code> is H, broadcast across rows; <code>Z<sup>[1]</sup></code> and <code>A<sup>[1]</sup></code> are m&times;H; <code>W<sup>[2]</sup></code> is H&times;1; and <code>&#375;</code> is m&times;1.</li>
        <li><b>Examples never mix.</b> The row index i passes through every operation untouched &mdash; nothing in the forward pass lets example 3 influence example 7 &mdash; and that is precisely what makes mini-batching a valid approximation rather than a different algorithm.</li>
        <li><b>Column j of W<sup>[1]</sup> is hidden unit j.</b> It is that unit's weight vector, and <code>W<sup>[1]</sup>[:,j]&#183;z + b<sup>[1]</sup>[j] = 0</code> is the line the unit draws. The fence figure in handout section 11 is plotted straight from those columns.</li>
        <li>One warning: many textbooks put examples in columns, and every transpose in the backpropagation derivation flips if you do. Mixing the two conventions halfway through is the commonest way to produce algebra that looks right and is not.</li>
    </ul>
    </p>
</details>


**11. An untrained network on the acceptance data has a cross-entropy cost
of 0.6721, and guessing costs log 2 = 0.6931. Why is that the right starting
value, and what would a very different one tell you?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Cross-entropy for a predictor that outputs a constant 0.5 on every example is <code>&minus;log 0.5 = log 2 = 0.6931</code>. A freshly initialised network has small random weights, so its pre-activations sit near zero and its outputs near 0.5: it starts, as it should, <b>knowing essentially nothing</b>.</li>
        <li>0.6721 is a shade below log 2 because the classes are not exactly balanced (the training base rate is 0.5471) and the random weights are not exactly zero &mdash; not because the network has learned anything.</li>
        <li>A starting cost far above log 2 means the initial weights are too large, so the network begins confidently wrong; a starting cost far below it means something has leaked, or the initialisation is not random. Either way it is a <b>free check</b> to run before training.</li>
    </ul>
    </p>
</details>


## Part 4 — Backpropagation


**12. Explain why neither perturbing each parameter in turn nor
differentiating the composed expression symbolically is used to get the
gradient, and state the two costs of backpropagation itself.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Perturbation</b> costs one forward pass per parameter. For the 301,066-parameter network of handout section 10 that is 301,066 forward passes for a <i>single</i> gradient step.</li>
        <li><b>Symbolic differentiation</b> of the whole composed expression produces something that repeats the same sub-expressions thousands of times. Backpropagation is the observation that those repeated sub-expressions can be computed <b>once each, right to left</b>: carrying <code>&delta;<sup>[l]</sup> = &part;J/&part;Z<sup>[l]</sup></code> gives layer l's weight gradients immediately <i>and</i> the same quantity for layer l&minus;1, so one backward sweep produces every gradient.</li>
        <li>Cost one, <b>arithmetic</b>: the backward pass does the same number of multiply-accumulates as the forward pass to within a factor of about two, so a gradient costs roughly what a prediction costs &mdash; which is what makes training feasible at all.</li>
        <li>Cost two, <b>memory</b>: every <code>A<sup>[l]</sup></code> from the forward pass must be stored until the backward pass reaches it, so memory grows with depth times batch size. That memory, not arithmetic, is usually what limits batch size in practice.</li>
    </ul>
    </p>
</details>


**13. Derive the output-layer gradient for a sigmoid output with
cross-entropy loss, and explain precisely what breaks if you pair the sigmoid
with mean squared error (MSE) instead.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Differentiating the cross-entropy with respect to the prediction gives <code>&part;J&#7522;/&part;&#375; = &minus;(1/m)[y/&#375; &minus; (1&minus;y)/(1&minus;&#375;)] = (1/m)&#183;(&#375; &minus; y)/[&#375;(1 &minus; &#375;)]</code>.</li>
        <li>The sigmoid's own derivative is <code>&sigma;&prime;(z) = e<sup>&minus;z</sup>/(1 + e<sup>&minus;z</sup>)<sup>2</sup> = &sigma;(z)(1 &minus; &sigma;(z)) = &#375;(1 &minus; &#375;)</code>. The chain rule multiplies the two, and the denominator <b>cancels exactly</b>: <code>&delta;<sup>[2]</sup> = &part;J/&part;Z<sup>[2]</sup> = (1/m)(&#375; &minus; y)</code>.</li>
        <li><b>This cancellation is why the pair is used.</b> With mean squared error the <code>&#375;(1 &minus; &#375;)</code> factor survives instead of cancelling, and that factor is near zero exactly when &#375; is near 0 or 1 &mdash; that is, when the network is <b>confidently wrong</b>, which is precisely the case where you most want a large gradient.</li>
        <li>So a sigmoid with squared error learns most slowly from the examples it gets most wrong. Cross-entropy removes the factor, and a confidently wrong example gets a gradient proportional to how wrong it is.</li>
    </ul>
    </p>
</details>


**14. Derive the weight and bias gradients for both layers from
&delta;<sup>[2]</sup>, checking the shapes, and explain why a ReLU unit with a
negative pre-activation passes nothing backward.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>From <code>Z<sup>[2]</sup> = A<sup>[1]</sup>W<sup>[2]</sup> + b<sup>[2]</sup></code>, differentiating componentwise gives <code>&part;J/&part;W<sup>[2]</sup> = (A<sup>[1]</sup>)<sup>T</sup>&delta;<sup>[2]</sup></code>, which is (H&times;m)(m&times;1) = H&times;1 &mdash; the shape of W<sup>[2]</sup>, as it must be. The bias appears once per example, so <code>&part;J/&part;b<sup>[2]</sup> = &Sigma;&#7522; &delta;<sup>[2]</sup>&#7522;</code>.</li>
        <li>Stepping down, A<sup>[1]</sup> affects the cost only through Z<sup>[2]</sup>, so <code>&part;J/&part;A<sup>[1]</sup> = &delta;<sup>[2]</sup>(W<sup>[2]</sup>)<sup>T</sup></code> (m&times;H), and passing back through the elementwise activation gives <code>&delta;<sup>[1]</sup> = (&delta;<sup>[2]</sup>(W<sup>[2]</sup>)<sup>T</sup>) &#8857; g&prime;(Z<sup>[1]</sup>)</code>. Then identically to the layer above, <code>&part;J/&part;W<sup>[1]</sup> = X<sup>T</sup>&delta;<sup>[1]</sup></code> (n&times;H) and <code>&part;J/&part;b<sup>[1]</sup> = &Sigma;&#7522; &delta;<sup>[1]</sup>&#7522;</code>.</li>
        <li>In words: <b>a weight's gradient is how wrong the layer above it was, multiplied by how much this weight contributed to that layer's input</b>. Everything above is that sentence in matrix form.</li>
        <li>For the ReLU, <code>g&prime;(z) = 1[z &gt; 0]</code>, an indicator. A unit whose pre-activation was negative contributed <b>nothing forward</b>, so it is right that it receives nothing backward &mdash; the elementwise product with g&prime; zeroes its entry in &delta;<sup>[1]</sup>. (At z = 0 the derivative does not exist; implementations pick 0 or 1 and it makes no measurable difference, since exact zeros essentially never occur.)</li>
    </ul>
    </p>
</details>


**15. State the gradient check, explain why the two-sided version is used
rather than the one-sided one, and give the thresholds for believing the
result.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Compare the analytic gradient against <code>[J(&theta; + h&#183;e&#7522;) &minus; J(&theta; &minus; h&#183;e&#7522;)] / (2h)</code> for each parameter, with h = 10&#8315;&#8309; in double precision.</li>
        <li>Expanding both terms as Taylor series, the O(h) and O(h&#178;) error terms cancel, leaving an error of <b>O(h&#178;)</b> against O(h) for the one-sided difference. That extra order is what makes rounding, not truncation, the limiting error &mdash; and so what makes agreement to eight decimal places meaningful.</li>
        <li><b>Below about 10&#8315;&#8310; relative disagreement, believe the gradient; above about 10&#8315;&#8308;, there is a bug.</b> Notebook 01 measures a worst relative disagreement of 1.97&times;10&#8315;&#8312; and a median of 7.83&times;10&#8315;&#185;&#185; over all 21 partial derivatives of a small network.</li>
        <li>Run it on a small network with a handful of examples whenever you write a backward pass by hand. A wrong analytic gradient <b>does not raise an exception</b> &mdash; it trains badly, slowly, or to the wrong place, and looks exactly like a modelling problem.</li>
    </ul>
    </p>
</details>


## Part 5 — More than two classes


**16. Define the softmax, state what it reduces to for two classes, and
explain why implementations subtract the maximum score before
exponentiating.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>softmax(z)<sub>k</sub> = e<sup>z<sub>k</sub></sup> / &Sigma;<sub>j</sub> e<sup>z<sub>j</sub></sup></code> over the K output units: positive by construction, summing to one, and monotone in each z&#8342;. The loss is categorical cross-entropy, which for a one-hot target is just <code>&minus;log &#375;<sub>c</sub></code> for the true class c.</li>
        <li>For K = 2 it reduces to the sigmoid &mdash; the two are the same construction at different K, which is why the binary and multiclass derivations agree.</li>
        <li>In practice the exponentials are computed as <code>e<sup>z<sub>k</sub> &minus; max<sub>j</sub> z<sub>j</sub></sup></code>. This changes <b>nothing mathematically</b>, because the constant factor cancels between numerator and denominator, but it prevents an <b>overflow</b> that is otherwise easy to hit: a logit of 800 exponentiates to infinity in double precision and turns the whole distribution into not-a-number.</li>
    </ul>
    </p>
</details>


**17. Derive the softmax-with-cross-entropy output gradient from the softmax
Jacobian, and say why getting the same answer as the binary case is not a
coincidence.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The Jacobian is <code>&part;&#375;&#8342;/&part;z&#8343; = &#375;&#8342;(&delta;&#8342;&#8343; &minus; &#375;&#8343;)</code>, where &delta;&#8342;&#8343; is 1 when k = l and 0 otherwise.</li>
        <li>With <code>L = &minus;&Sigma;&#8342; y&#8342; log &#375;&#8342;</code>, the chain rule gives <code>&part;L/&part;z&#8343; = &minus;&Sigma;&#8342; (y&#8342;/&#375;&#8342;)&#183;&#375;&#8342;(&delta;&#8342;&#8343; &minus; &#375;&#8343;) = &minus;&Sigma;&#8342; y&#8342;(&delta;&#8342;&#8343; &minus; &#375;&#8343;) = &minus;y&#8343; + &#375;&#8343;&Sigma;&#8342; y&#8342; = &#375;&#8343; &minus; y&#8343;</code>, using &Sigma;&#8342; y&#8342; = 1 for a one-hot target. Every &#375;&#8342; in the denominator cancels, exactly as in the binary case.</li>
        <li>It is not a coincidence: <b>the loss is chosen as the one whose derivative cancels the output non-linearity's</b>, and sigmoid-with-cross-entropy is the K = 2 instance of softmax-with-cross-entropy.</li>
        <li>The practical consequence is that <b>nothing downstream changes</b>: every formula of question 14 applies unchanged, with &delta;<sup>[2]</sup> now m&times;K instead of m&times;1.</li>
    </ul>
    </p>
</details>


**18. On the digits, softmax alone scores 0.9324, one hidden layer of 32
scores 0.9602 and one of 64 scores 0.9611, with seed-to-seed standard
deviations around 0.005. What should you conclude, and why does the acceptance
problem behave so differently?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Multiclass logistic regression with <b>no hidden layer at all</b> is within 3.5 points of the best network on the table (two layers of 64, at 0.9676). The first hidden layer is worth about 2.8 points and is clearly real.</li>
        <li>Doubling the layer from 32 units to 64 is worth <b>0.09 points</b> against a seed-to-seed spread of about 0.5 points &mdash; a fifth of the noise, and therefore <b>nothing</b>. Reporting it as an improvement would be reporting one draw of the random seed.</li>
        <li>The difference from the acceptance problem, where the same step was worth 39 points, is <b>structural, not a matter of tuning</b>. The digits are close to linearly separable in pixel space because classes differ in <i>which pixels are dark</i>, and a weighted sum of pixels captures most of that. The acceptance rule depended on two measurements <i>in combination</i>, which is exactly what a weighted sum cannot express.</li>
        <li><b>A hidden layer is not always the answer</b>, and the practical rule is to fit the linear model first so that you know what the hidden layer has to beat.</li>
    </ul>
    </p>
</details>


## Part 6 — Activations, and the vanishing gradient


**19. Show that a network with identity activations collapses to a single
linear layer, and explain why that collapse is worth remembering when
debugging a real network.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With g the identity, <code>Z<sup>[2]</sup> = (XW<sup>[1]</sup> + b<sup>[1]</sup>)W<sup>[2]</sup> + b<sup>[2]</sup> = X(W<sup>[1]</sup>W<sup>[2]</sup>) + (b<sup>[1]</sup>W<sup>[2]</sup> + b<sup>[2]</sup>)</code>, which is a single linear layer with weights <code>W<sup>[1]</sup>W<sup>[2]</sup></code> and bias <code>b<sup>[1]</sup>W<sup>[2]</sup> + b<sup>[2]</sup></code>. Any number of linear layers composes to one, no matter how deep.</li>
        <li>So the <b>non-linearity is the entire reason depth buys anything</b>. Depth without it is an expensive way to write a matrix product.</li>
        <li>The debugging value is that a network whose activations have <b>all saturated</b> (every tanh output pinned at &plusmn;1) or <b>all died</b> (every ReLU output zero) has effectively performed the same collapse, and will behave like a much simpler model than the one you think you are training.</li>
    </ul>
    </p>
</details>


**20. Prove that the sigmoid's derivative never exceeds one quarter, and
work out what that implies for the backward recursion over L layers.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>From question 13, <code>&sigma;&prime;(z) = &sigma;(z)(1 &minus; &sigma;(z))</code>. Writing s = &sigma;(z) &isin; (0,1), the function s(1 &minus; s) is a downward parabola with its maximum at s = &frac12;, so <code>max<sub>z</sub> &sigma;&prime;(z) = &frac12; &times; &frac12; = &frac14;</code>, attained at z = 0 and nowhere else.</li>
        <li>The backward recursion <code>&delta;<sup>[l]</sup> = (&delta;<sup>[l+1]</sup>(W<sup>[l+1]</sup>)<sup>T</sup>) &#8857; g&prime;(Z<sup>[l]</sup>)</code> multiplies by two things per layer: the weight matrix and the activation's derivative. Initialisation is <i>chosen</i> so the weight matrix contributes a factor of about 1, which leaves the derivative in charge &mdash; and it can only shrink, giving <code>&#8214;&delta;<sup>[l]</sup>&#8214; &#8818; &frac14;&#8214;&delta;<sup>[l+1]</sup>&#8214;</code>.</li>
        <li>Over L layers this compounds to roughly <b>4<sup>L</sup></b>. At initialisation the bound is close to <b>tight rather than loose</b>, because the pre-activations sit near zero &mdash; exactly where &sigma;&prime; takes its maximum.</li>
        <li>This is the number to carry out of the lesson: <b>a sigmoid layer divides the gradient by about four</b>.</li>
    </ul>
    </p>
</details>


**21. Across eight seeds the per-layer shrinkage of a six-layer sigmoid
network measures 3.90, 3.97, 4.14, 4.05, 3.92, 4.31, 4.06, 3.86, while the
end-to-end ratio has a median of 3,547 and ranges from 2,734 to 4,607. Explain
why quoting &ldquo;the gradient shrinks 4,607-fold&rdquo; would be
dishonest.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>4,607 is the <b>largest of eight draws</b>, and quoting it reports one sample extremum as though it were a law. It is the same error as question 4's brute-force line: selecting the maximum of a noisy quantity and then presenting that maximum as the quantity.</li>
        <li>The <b>per-layer factor is the property</b>. It has a mechanism behind it &mdash; &sigma;&prime; &le; &frac14; &mdash; and it is tight across seeds, all eight landing between 3.9 and 4.3. That is a number you can predict in advance and check.</li>
        <li>The <b>end-to-end number is what that property compounds to</b>, and it inherits six layers' worth of scatter: it varies by a factor of 1.7 across the same eight seeds, which is why only its order of magnitude means anything. Quote the median with its range, or quote the per-layer factor and let the reader raise it to the sixth power.</li>
        <li>The depth sweep is the honest version of the claim: medians of 2.3, 9.5, 169.6, 3,552.9 and 46,250 at 1, 2, 4, 6 and 8 hidden layers &mdash; geometric growth over four orders of magnitude, always within a factor of two of 4<sup>depth</sup> and always slightly below it, because the weight matrices give back a little of what the derivative takes.</li>
    </ul>
    </p>
</details>


**22. On the same six-layer architecture, tanh finishes at 0.9750 and the
ReLU at 0.9611, while the sigmoid never leaves 0.1000. Why is tanh beating the
ReLU consistent with the argument rather than a surprise?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>0.1000 is <b>exact chance on ten classes</b>: after eighty epochs the sigmoid network has learned nothing at all, because the gradient reaching its first layer is three and a half orders of magnitude smaller than the gradient at its last.</li>
        <li>tanh is <b>every bit as much a squashing function as the sigmoid</b> &mdash; it saturates at &plusmn;1 in exactly the same way. It is fine because <b>its derivative peaks at 1 rather than &frac14;</b>, so the backward recursion is not systematically divided by anything. What kills the sigmoid is not squashing; it is <i>where its derivative is bounded</i>.</li>
        <li>The ReLU's advantages &mdash; cheaper to compute, no saturation for large positive input &mdash; are real, and &ldquo;use ReLU&rdquo; remains good default advice. But on six layers of 32 units they simply do not show up, and tanh reached 0.85 in 3 epochs against the ReLU's 9.</li>
        <li>Reporting this the other way round, or quietly dropping the tanh row, would be teaching a <b>slogan instead of a mechanism</b>. The mechanism predicts both results; the slogan predicts only one of them.</li>
    </ul>
    </p>
</details>


**23. Most students conclude from the vanishing gradient that the ReLU has
no gradient problem. Explain why that reasoning is sound as far as it goes,
what it misses, and how large the measured cost turned out to be.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The reasoning really is sound in its own terms: the ReLU's derivative is exactly 1 on the active side, and it does not saturate however large the positive input grows. Nothing about the sigmoid's failure applies to it.</li>
        <li>What it misses is the other side. A ReLU unit whose pre-activation is negative for <i>every</i> training example outputs zero for all of them, therefore receives zero gradient from all of them, for ever. It is <b>dead</b>, and no further training revives it &mdash; the gradient that would move it is the one it cannot receive. A single large step is enough to knock a bias far enough negative to cause this.</li>
        <li>The measured cost is <b>smaller than the drama suggests</b>: at a learning rate of 1.0, 33 of 64 units are dead and validation accuracy is 0.9583 against a best of 0.9667 &mdash; less than one point, because the survivors absorb the work.</li>
        <li>So dead units are usually <b>wasted capacity rather than catastrophe</b>, which is exactly why the failure is easy to miss: nothing in the training curve announces it. A layer of 64 that is half dead is a layer of 31, and if that layer were your bottleneck you would be tuning everything except the thing that is wrong.</li>
    </ul>
    </p>
</details>


## Part 7 — Initialisation


**24. Lesson 3 started gradient descent from w = 0 and that was correct.
Explain what goes wrong when the habit is carried into a network, why the
usual symmetry argument understates it, and derive the cost the all-zero
network converges to.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>For linear and logistic regression the cost is convex and the origin is as good a start as any. The <b>usual symmetry argument</b> against it in a network is that identical hidden units compute identical outputs, receive identical gradients, take identical steps and stay identical for ever. That is correct, and it applies to <i>any</i> initialisation giving two units the same weights.</li>
        <li>For <b>all</b> weights zero something stronger happens. Since W<sup>[2]</sup> = 0, <code>&delta;<sup>[1]</sup> = (&delta;<sup>[2]</sup>(W<sup>[2]</sup>)<sup>T</sup>) &#8857; g&prime;(Z<sup>[1]</sup>) = 0</code>, so W<sup>[1]</sup> and b<sup>[1]</sup> receive exactly zero gradient; and since <code>A<sup>[1]</sup> = ReLU(0) = 0</code>, <code>&nabla;W<sup>[2]</sup> = (A<sup>[1]</sup>)<sup>T</sup>&delta;<sup>[2]</sup></code> is zero too. <b>Every parameter in the network is frozen except b<sup>[2]</sup></b> &mdash; it is not slow, it is stationary.</li>
        <li>The network is therefore a constant predictor, and b<sup>[2]</sup> converges to the value making that constant the training base rate &#563; = 0.5471. Its cost converges to the <b>entropy of the label distribution</b>: <code>&minus;[0.5471&#183;log 0.5471 + 0.4529&#183;log 0.4529] = 0.6887</code>, and notebook 01 measures the final cost as 0.6887 &mdash; not approximately, because the mechanism is exact.</li>
        <li>Counting distinct columns of W<sup>[1]</sup> afterwards gives <b>1 of 32</b> from the zero start against 32 of 32 from a random one. Random initialisation is not a heuristic that happens to help; it is what makes the units <i>different problems to solve</i>.</li>
    </ul>
    </p>
</details>


**25. Derive why Glorot initialisation wants Var(w) = 1/n while He
initialisation wants 2/n, and say which one belongs with the ReLU.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A unit computes <code>z = &Sigma;&#7522; w&#7522;a&#7522;</code> over n inputs. Taking the w&#7522; independent of the a&#7522;, mutually independent and zero-mean, <code>Var(z) = n&#183;Var(w)&#183;E[a&#178;]</code>.</li>
        <li>If the incoming activations are also zero-mean then E[a&#178;] = Var(a), so each layer multiplies the variance by <code>n&#183;Var(w)</code>. Anything other than 1 compounds <b>geometrically with depth</b> &mdash; collapsing to nothing or exploding &mdash; which forces <code>Var(w) = 1/n</code> (<b>Glorot / Xavier</b>).</li>
        <li>The ReLU changes the derivation in exactly one place: it <b>zeroes the negative half</b>, so for symmetric z, <code>E[a&#178;] = &frac12;Var(z)</code>. Half the variance is thrown away at every layer, and compensating for that factor of two gives <code>Var(w) = 2/n</code> (<b>He</b>).</li>
        <li>So the extra factor of 2 is not a tuning constant &mdash; it is the price of the ReLU's rectification, and it is what <code>kernel_initializer="he_normal"</code> means in Keras. Use He with the ReLU, Glorot with tanh or the sigmoid.</li>
    </ul>
    </p>
</details>


**26. In an eight-layer tanh network the per-layer output standard
deviation runs 0.8858 &rarr; 0.9463 &rarr; 0.9472 for N(0,1) weights,
0.0412 &rarr; 0.0000 &rarr; 0.0000 for N(0,0.01), and 0.4063 &rarr; 0.3066
&rarr; 0.2102 for Glorot. Diagnose the two failures and say why the first
column is not an explosion.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Too small collapses.</b> By layer 4 the signal is indistinguishable from zero, and nothing downstream can recover what is no longer there &mdash; the forward pass has multiplied the variance by well under 1 at each of eight layers.</li>
        <li><b>Too large does not explode</b>, because tanh cannot exceed 1: bounding the activation bounds the standard deviation. What happens instead is <b>saturation</b> &mdash; 73% of the last layer's units sit past |a| &gt; 0.99, where the derivative is indistinguishable from zero. A saturated layer passes signal forward and <b>nothing backward</b>, which is question 20's failure arriving from the other direction.</li>
        <li>Reading the standard deviation alone would make the N(0,1) column look healthy, which is the trap: 0.9472 at layer 8 is a distribution pinned at &plusmn;1, not a well-spread one. The diagnostic to look at is the <i>fraction saturated</i>, not the spread.</li>
        <li><b>Glorot roughly holds</b>, losing about half its spread over eight layers. That is what &ldquo;preserving the signal&rdquo; looks like in practice &mdash; unimpressive on its own, and decisive against three orders of magnitude of collapse or near-total saturation.</li>
    </ul>
    </p>
</details>


## Part 8 — Optimisation in practice


**27. Learning rates of 0.001, 0.01, 0.1, 0.5 and 2.0 give validation
accuracies of 0.5574, 0.9398, 0.9556, 0.9685 and 0.1009. Describe the three
regimes, and name the predictable mistake made at the low end.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Too large</b> (2.0): the steps overshoot every minimum they approach, the training loss sits at 2.3222 and accuracy is 0.1009 &mdash; exact chance on ten classes.</li>
        <li><b>Too small</b> (0.001): the loss is still falling when the epochs run out, leaving a final training loss of 1.9550 and accuracy 0.5574 with a standard deviation of 0.0794 across seeds. The useful band between the two extremes is only about one and a half orders of magnitude wide.</li>
        <li><b>The predictable mistake is diagnosing the second case as a capacity problem.</b> A network underfitting because &alpha; is too small looks <i>exactly</i> like a network that is too small &mdash; training and validation accuracy both low, both still improving &mdash; and the reasonable response, adding units and layers, makes it slower without making it better. The reasoning is sound as far as it goes: low training accuracy really is the classic signature of underfitting. The cause is just in the optimiser, not the architecture.</li>
        <li>Hence the rule: <b>vary the learning rate over orders of magnitude before touching the architecture</b>. And note that a rate stable early can be unstable later &mdash; at &alpha; = 0.5 notebook 01's four-unit network reaches a training cost of 0.2564 at epoch 268 and then <i>climbs</i> to 0.8737 by epoch 400, with five runs in eight ending above their own minimum.</li>
    </ul>
    </p>
</details>


**28. Define full-batch, stochastic and mini-batch gradient descent, and
state the three trade-offs mini-batches sit in the middle of.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Full-batch</b> gradient descent computes &nabla;J on all m examples per step. <b>Stochastic gradient descent (SGD)</b> in its literal form uses one example per step. <b>Mini-batch SGD</b> &mdash; which is what everyone means by SGD in practice &mdash; uses a few dozen to a few hundred; this lesson uses 32 or 64 throughout.</li>
        <li>Trade-off one, <b>estimate quality</b>: the mini-batch gradient is noisy but <i>unbiased</i>, so many cheap approximate steps beat few exact ones.</li>
        <li>Trade-off two, <b>arithmetic</b>: a batch of 32 vectorises into matrix products, where a single example wastes most of the hardware.</li>
        <li>Trade-off three, <b>the noise itself helps</b>, jostling the parameters out of the shallow local minima of question 8. Note that batch size and learning rate interact &mdash; a larger batch gives a less noisy gradient and tolerates a larger &alpha; &mdash; so they should not be tuned independently.</li>
    </ul>
    </p>
</details>


**29. Plain gradient descent, momentum and adaptive moment estimation
(Adam) score 0.9389, 0.9349 and 0.9485 on the acceptance data, with standard
deviations 0.0184, 0.0094 and 0.0072 and worst runs of 0.9027, 0.9173 and
0.9360. Read the table, and say what momentum bought.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Momentum</b> keeps a running average <code>v &larr; &beta;v + (1&minus;&beta;)&nabla;J</code> with &beta; = 0.9 and steps along v: consistent directions accumulate, oscillating ones cancel. It is the direct repair for the stretched-valley problem lesson 3 described through the condition number. <b>Adam</b> adds a per-parameter step size, dividing each coordinate's step by a running estimate of that coordinate's own gradient magnitude, which is what makes it forgiving of a badly chosen global &alpha;.</li>
        <li><b>Read the spread before the mean.</b> Adam's mean is about a point above plain descent, which is barely more than the spread; its <i>worst run is more than three points above</i>, and its standard deviation is less than half. What Adam bought is not a lower floor but the <b>disappearance of the bad case</b> &mdash; and insuring against the bad case is exactly what question 8's five restarts were doing by hand. Adam in one run matches what plain descent needed five to reach.</li>
        <li><b>Momentum, on this problem, did nothing at all</b>: a fraction of a point below plain descent, well inside either method's spread. That is worth reporting rather than quietly dropping. Momentum is a good default, not a guarantee, and <b>a comparison in which every row improves on the last is usually a comparison that has been curated.</b></li>
    </ul>
    </p>
</details>


## Part 9 — Regularisation, and knowing whether it worked


**30. A network with 301,066 parameters trained on 300 digits reaches
training accuracy 1.0000 while validation accuracy holds at 0.9472. Explain
which of those two facts is a problem, and what the validation loss does that
the validation accuracy cannot show.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>The network memorises the training set</b>, and with 1,004 parameters per training example that is unsurprising: there is more than enough freedom to store the answers outright.</li>
        <li><b>And it generalises anyway</b>, holding validation accuracy at 0.9472 with a best of 0.9500. Classical bias&ndash;variance reasoning from lesson 5 does not lead you to expect this, and it is nevertheless how large networks behave. Taking it seriously is an open research question; taking it as licence to stop validating would be a serious mistake.</li>
        <li>What <i>does</i> degrade is the validation <b>loss</b>, which bottoms out early and then climbs for the rest of training while validation accuracy stays flat. The network is not getting more answers wrong; it is getting <b>steadily more confident about the ones it already has wrong</b>.</li>
        <li>Accuracy cannot see that, because it only counts which side of 0.5 a prediction falls on, and cross-entropy can, because it reads the confidence. That is the argument for <b>early-stopping on the loss rather than on accuracy</b>.</li>
    </ul>
    </p>
</details>


**31. Define early stopping, L2 regularisation and dropout, including what
dropout does at prediction time and why the survivors are rescaled.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Early stopping</b> monitors a validation metric and keeps the weights from the best epoch. It is the cheapest regulariser there is, needs no hyperparameter beyond a patience, and is the one to reach for first.</li>
        <li><b>L2 regularisation</b> adds <code>(&lambda;/2)&#8214;W&#8214;&#178;</code> to the cost, which adds <code>&lambda;W</code> to every weight gradient: each step shrinks the weight slightly toward zero before the data's gradient moves it. Identical in form to Ridge in lesson 3, and in this context usually called <b>weight decay</b>.</li>
        <li><b>Dropout</b> deletes each unit independently with probability p on every training batch and rescales the survivors by <code>1/(1&minus;p)</code>, so the <i>expected</i> input to the next layer is unchanged and the network does not see a systematically smaller signal in training than in use. At <b>prediction time nothing is dropped</b> and no rescaling is applied.</li>
        <li>The usual explanation is that a unit cannot rely on any particular other unit being present, so the layer cannot build fragile co-adaptations; an equivalent reading is that it trains an ensemble of exponentially many thinned networks sharing weights, which connects it to lesson 7's bagging.</li>
    </ul>
    </p>
</details>


**32. The regularisation table reports 0.9411, 0.9483, 0.9478 and 0.9511
for early stopping alone, plus dropout, plus L2, and both &mdash; with
standard deviations of 0.0069, 0.0038, 0.0011 and 0.0045 over five seeds on
360 test examples. Explain why this experiment cannot rank dropout against
L2, and what it <i>can</i> conclude.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>All three interventions land between <b>0.7 and 1.0 points</b> above early stopping alone, against a seed-to-seed spread of 0.1 to 0.7 points. That is one to two standard deviations: <b>enough to say they helped, nowhere near enough to rank them</b>.</li>
        <li>Dropout and L2 differ by 0.05 points &mdash; 0.9483 against 0.9478 &mdash; which is an order of magnitude inside their own seed-to-seed variation. On a single run with one seed you could comfortably have measured the three in <b>any order</b>.</li>
        <li>The test set itself sets a second floor on resolution: 360 examples at around 95% accuracy give a standard error of roughly one point, so even a perfectly repeatable difference of a half-point could not be established from this test set at all. Ranking them would need more seeds, more test data, or both.</li>
        <li>What the table <i>can</i> conclude is in the column the accuracies cannot speak to: <b>epochs run</b>. L2 kept the validation loss improving for 198.4 epochs against 38.8 for early stopping alone &mdash; roughly four times as long &mdash; a real qualitative difference in a comparison whose headline numbers are not separable. Reporting four bare numbers with no spread beside them would have supported a confident sentence about which regulariser is best, and that sentence would have been unfounded.</li>
    </ul>
    </p>
</details>


**33. Going from 300 training examples to 1,077 moves test accuracy from
0.9463 to 0.9787, while the best regulariser at 300 examples was worth +1.0
points. What follows, and what is the catch?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>More data is worth <b>+3.2 points</b> here against the best regulariser's <b>+1.0</b> &mdash; a factor of three &mdash; and the gain is monotone across the whole sweep: 0.9028 at 100 examples, 0.9426 at 200, 0.9694 at 500, 0.9722 at 800, 0.9787 at 1,077.</li>
        <li>The practical order follows: before spending an afternoon tuning dropout rates and penalty strengths, ask whether more labelled data is available, because it typically buys more and buys it more reliably.</li>
        <li><b>The catch is that it is not a hyperparameter.</b> Unlike every method in the previous question, more data is not something you can tune your way to &mdash; it has to be collected or labelled, at a cost lesson 2 spent an hour on. Where the data is fixed, the regularisers are what you have.</li>
    </ul>
    </p>
</details>


## Part 10 — Reading the score honestly, and choosing


**34. State the identity connecting a classifier's agreement with the true
acceptance rule to its measured accuracy against the rig's labels, apply it to
the 16-unit network, and say what it implies for a real dataset.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>If a classifier agrees with the true rule on a fraction q of units and the rig independently records the wrong verdict with probability e, the two agree exactly when both are right or both are wrong: <code>measured accuracy = q(1 &minus; e) + (1 &minus; q)e</code>. At q = 1 this gives <code>1 &minus; e = 0.97</code>, the ceiling quoted throughout the lesson.</li>
        <li>The 16-unit network agrees with the true acceptance rule on 97.60% of units, so its expected measured accuracy is <code>0.9760 &times; 0.97 + 0.0240 &times; 0.03 = 0.9474</code>, against <b>0.9475 observed</b>. Across the whole width sweep the largest discrepancy is 0.0044, the residual being that the model's errors are not quite independent of the rig's &mdash; both concentrate near the boundary.</li>
        <li>So the network learned the true boundary to within 2.4%, and is <i>scored</i> at 0.9475: the 2.9-point difference is a property of the <b>measuring instrument</b>, not of the model.</li>
        <li><b>On any real dataset only the lower number exists.</b> Nobody publishes the true rule, so there is no way from inside the data to tell how much of the shortfall is yours and how much is the labels'. That is the honest reading of every headline score in this lesson, and the reason to be sceptical of the last point or two of anyone else's.</li>
    </ul>
    </p>
</details>


**35. Across the width sweep, a trained 3-unit network scores 0.9421 while
the best regular 3-sided fence scores 0.8632, yet at 16 units the fence
(0.9667) has overtaken the trained network (0.9475). Explain both crossings.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>The polygon is a floor, not a ceiling.</b> The fence curve asks how well H lines can enclose the circle as an <i>intersection of half-planes</i> &mdash; one convex polygon. But three lines cut the plane into seven regions, not one triangle, and the output unit takes a weighted vote over which side of each line a point falls, which can select shapes an intersection cannot. So the trained network beats the best regular polygon of the same number of sides, by 7.9 points at three units.</li>
        <li><b>Past six units the binding constraint changes.</b> The fence curve keeps climbing toward 0.97 as more sides become available; the trained network does not follow, settling just below 0.95. The lines are there and <b>gradient descent does not put them to work</b> &mdash; that gap is the optimiser, not the capacity, which is what Adam then partly closes (0.9485, with the worst of five runs at 0.9360).</li>
        <li>The overall shape of the sweep is the point worth carrying: one unit scores what the best single line scores (0.6512 against 0.6491), the step from two units to three is worth <b>20 points</b>, and everything from three units to thirty-two is worth <b>0.5</b>. Almost everything arrives in the first three lines.</li>
    </ul>
    </p>
</details>


**36. A colleague brings you a network that sits at chance after training,
a second that has high training accuracy and low validation accuracy, and a
third whose training and validation accuracy are both low and still climbing.
Prescribe an order of investigation for each, and say when a hidden layer is
worth its cost at all.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>At chance:</b> check the <b>learning rate</b> before the architecture &mdash; a rate too large sits at exactly chance, as 0.1009 on ten classes did. If the rate is sane and the network is deep, check the <b>activation</b> next (a six-layer sigmoid stack never leaves chance), then the <b>initialisation</b> (all-zero weights freeze everything but one output bias). If the backward pass was written by hand, <b>gradient-check it</b> before anything else.</li>
        <li><b>High training, low validation:</b> this is overfitting. Early stopping first, because it is free; then more data, which bought 3.2 points where the best regulariser bought 1.0; then dropout or L2. And early-stop on the <b>loss</b>, since accuracy cannot see growing overconfidence.</li>
        <li><b>Both low and still climbing:</b> this is underfitting, and the first suspect is the optimiser, not the architecture &mdash; &alpha; too small, or too few epochs. Vary the learning rate over orders of magnitude before adding units, or you will make the network slower without making it better.</li>
        <li><b>When a hidden layer earns its cost:</b> when the boundary depends on inputs <i>in combination</i>, as Meridian's two-tolerance rule did (0.55 &rarr; 0.94). When the inputs vote roughly independently, try the linear model first &mdash; on the digits it lost only 3.5 points to the best network here, at a fraction of the parameters and none of the tuning.</li>
    </ul>
    </p>
</details>
